In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

df = pd.read_csv("../data/creditcard_with_hour.csv")
print(df.shape)

(284807, 32)


In [2]:
# Sort by Time so rolling windows make sense
df = df.sort_values("Time").reset_index(drop=True)

# Feature 1: Time gap since last transaction (in seconds)
df["time_gap"] = df["Time"].diff().fillna(0)

# Feature 2: Rolling transaction count in last ~1 hour (3600 seconds)
# Approximation using row-based window (last 10 transactions as proxy)
df["rolling_count_10"] = df["Amount"].rolling(window=10, min_periods=1).count()

# Feature 3: Rolling average amount (last 10 transactions)
df["rolling_avg_amt_10"] = df["Amount"].rolling(window=10, min_periods=1).mean()

# Feature 4: Rolling std of amount (last 10 transactions)
df["rolling_std_amt_10"] = df["Amount"].rolling(window=10, min_periods=1).std().fillna(0)

# Feature 5: Amount deviation from rolling mean
df["amount_deviation"] = df["Amount"] - df["rolling_avg_amt_10"]

# Feature 6: Is this a round-number amount? (e.g., 100.00, 200.00)
df["is_round_amount"] = (df["Amount"] % 1 == 0).astype(int)

# Feature 7: Amount bucket
bins = [-1, 1, 10, 100, 500, 1000, float("inf")]
labels = [0, 1, 2, 3, 4, 5]
df["amount_bucket"] = pd.cut(df["Amount"], bins=bins, labels=labels).astype(int)

# Feature 8: Hour interaction with amount
df["hour_x_amount"] = df["Hour"] * df["Amount"]

print("New features added:")
new_feats = ["time_gap", "rolling_count_10", "rolling_avg_amt_10",
             "rolling_std_amt_10", "amount_deviation", "is_round_amount",
             "amount_bucket", "hour_x_amount"]
print(df[new_feats].describe())

New features added:
            time_gap  rolling_count_10  rolling_avg_amt_10  \
count  284807.000000     284807.000000       284807.000000   
mean        0.606699          9.999842           88.350717   
std         1.053380          0.031633           81.687215   
min         0.000000          1.000000            0.010000   
25%         0.000000         10.000000           40.460000   
50%         0.000000         10.000000           66.464000   
75%         1.000000         10.000000          109.600000   
max        32.000000         10.000000         2712.061000   

       rolling_std_amt_10  amount_deviation  is_round_amount  amount_bucket  \
count       284807.000000     284807.000000    284807.000000  284807.000000   
mean           149.663736         -0.001098         0.248751       1.781726   
std            199.243275        236.618128         0.432290       0.985853   
min              0.000000      -2702.382000         0.000000       0.000000   
25%             50.781483 

In [3]:
# Drop original Time column (we have Hour now)
df = df.drop(columns=["Time"])

# Mean-center Amount and Hour (same as original blog)
df["Amount"] = df["Amount"] - df["Amount"].mean()
df["Hour"] = df["Hour"] - df["Hour"].mean()

In [4]:
# IMPORTANT: Split BEFORE applying SMOTE
# Never apply SMOTE to test set — that would be data leakage

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"Fraud in train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Fraud in test:  {y_test.sum()} ({y_test.mean()*100:.2f}%)")

Train size: (227845, 38)
Test size:  (56962, 38)
Fraud in train: 394 (0.17%)
Fraud in test:  98 (0.17%)


In [5]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Train size: {X_train_sm.shape}")
print(f"Class balance: {pd.Series(y_train_sm).value_counts().to_dict()}")

After SMOTE — Train size: (454902, 38)
Class balance: {0: 227451, 1: 227451}


In [6]:
joblib.dump((X_train_sm, y_train_sm, X_test, y_test), "../data/processed_data.pkl")
joblib.dump(X_train.columns.tolist(), "../data/feature_names.pkl")
print("Saved processed data to data/processed_data.pkl")

Saved processed data to data/processed_data.pkl
